# 05 - Benchmark final: split test, satu sesi GPU

Membuka split test SATU KALI, memakai konfigurasi terbaik tiap skenario menurut
`best.json`, lalu mengukur inferensi ketiga model berurutan pada GPU yang sama.

Aturan validitas: angka efisiensi (waktu latih, latency, peak memory) hanya sah
bila berasal dari satu hardware dan satu sesi. Menjalankan ulang sebagian
notebook ini di sesi lain lalu mencampur angkanya membatalkan perbandingan.
F1 tidak terpengaruh hardware.

Prasyarat: ketiga skenario sudah punya run dari `03a` sampai `03c`.

Jangan menjalankan notebook ini berulang kali untuk memilih hasil terbaik: itu
mengubah test menjadi validation set kedua.

In [ ]:
import json

import pandas as pd

from src.config import settings
from src.services.campaign import CampaignRunner

OUT_DIR = settings.default_out_dir
runner = CampaignRunner(out_dir=OUT_DIR)

hardware = runner.write_hardware()
print(json.dumps(hardware, indent=2, ensure_ascii=False))

## 1. Konfigurasi yang akan diuji

In [ ]:
best = json.loads((OUT_DIR / "best.json").read_text(encoding="utf-8"))
for skenario, entri in best.items():
    print(f"{skenario}: run #{entri['run_id']} | val F1 {entri['val_f1_macro']:.4f} "
          f"| {entri['config']}")

## 2. Jalankan benchmark

In [ ]:
hasil = runner.run_final()

## 3. Perbandingan test

In [ ]:
hasil["comparison"]

## 4. Benchmark inferensi

In [ ]:
hasil["benchmark"]

Ketiga skenario menjalankan forward pass encoder 110 juta parameter yang sama
saat inferensi, sehingga latency-nya diperkirakan berdekatan. Efisiensi RM-b dan
RM-c terletak pada jumlah parameter yang dilatih dan waktu latih, bukan pada
kecepatan prediksi. Kalau angkanya memang begitu, itu temuan yang harus
dinyatakan apa adanya di Bab 4, bukan disembunyikan.

## 5. Kriteria sukses

In [ ]:
hasil["criteria"]

Strategi ringan dianggap kompetitif bila memenuhi minimal dua dari tiga syarat:
selisih F1 tidak lebih dari 3 poin persentase, pengurangan trainable parameter
minimal 90%, dan pengurangan waktu latih minimal 50%.

## 6. Ekspor checkpoint final

In [ ]:
import shutil

settings.model_dir.mkdir(parents=True, exist_ok=True)
for nama in ("rma_best.pt", "rmb_best.pt", "rmc_best.pt"):
    sumber = OUT_DIR / "checkpoints" / nama
    if sumber.exists():
        tujuan = shutil.copy2(sumber, settings.model_dir / nama)
        print(f"  {nama}: {sumber.stat().st_size / 1024**2:.1f} MB -> {tujuan}")

## Ringkasan

Tabel di atas adalah angka resmi Bab 4. Seluruhnya diproduksi dalam satu sesi
pada hardware yang tercatat di `hardware.json` yang berdampingan dengannya.

Lanjut ke `06_analysis_export.ipynb`.